# 🔢 Extractor LMFDB — Grupos Transitivos de $S_7$ (modo ligero)

**Pipeline minimalista**: extrae los 7 grupos `7T1`…`7T7` con todos sus polinomios desde PostgreSQL de LMFDB (vía `lmfdb-lite`) y exporta a JSON.

**No imprime nada masivo en pantalla** — solo construye el repositorio en memoria y lo guarda en disco. `get_group()` queda definida pero no se ejecuta automáticamente; úsala solo si quieres consultar un grupo puntual.

El JSON resultante está pensado para subirse a una base de datos externa (ver la celda final con recomendaciones).


## Celda 1 — Instalar lmfdb-lite


In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'lmfdb-lite[pgbinary] @ git+https://github.com/roed314/lmfdb-lite.git'],
    capture_output=True, text=True
)
print('✅ lmfdb-lite instalado' if r.returncode == 0 else f'❌ Error: {r.stderr[-400:]}')


## Celda 2 — Conectar a LMFDB


In [ ]:
from lmf import db
print('✅ Conectado a PostgreSQL de LMFDB')


## Celda 3 — Parámetros


In [ ]:
DEGREE = 7     # Grado del polinomio
MAX_T  = 7     # Subgrupos transitivos de S7: 7T1 … 7T7
# MAX_POLYS = None  → descarga TODOS los polinomios de cada grupo
# MAX_POLYS = 50    → limita a 50 por grupo (más rápido para pruebas)
MAX_POLYS = None
print(f'Grado: {DEGREE}  |  Grupos: {DEGREE}T1 … {DEGREE}T{MAX_T}')
print(f'Polinomios por grupo: {"TODOS" if MAX_POLYS is None else MAX_POLYS}')


## Celda 4 — Helpers


In [ ]:
def coeffs_to_poly_str(coeffs, var='x'):
    """
    Convierte [a0, a1, …, an] → string legible del polinomio mónico.
    LMFDB almacena en orden ascendente; el último coeficiente siempre es 1.
    """
    if not coeffs:
        return 'N/A'
    n = len(coeffs) - 1
    terms = []
    for i in range(n, -1, -1):
        c = coeffs[i]
        if c == 0:
            continue
        if i == 0:
            terms.append(str(c))
        elif i == 1:
            if   c ==  1: terms.append(var)
            elif c == -1: terms.append(f'-{var}')
            else:         terms.append(f'{c}*{var}')
        else:
            if   c ==  1: terms.append(f'{var}^{i}')
            elif c == -1: terms.append(f'-{var}^{i}')
            else:         terms.append(f'{c}*{var}^{i}')
    if not terms:
        return '0'
    poly = terms[0]
    for tk in terms[1:]:
        poly += (' - ' + tk[1:]) if tk.startswith('-') else (' + ' + tk)
    return poly


def _fmt_group_ref(s):
    """Formatea una referencia a grupo transitivo desde lista [n, t] o similar."""
    if isinstance(s, (list, tuple)) and len(s) >= 2:
        # puede ser [[n,t], multiplicidad] o [n,t]
        inner = s[0] if isinstance(s[0], (list, tuple)) else s
        return f'{inner[0]}T{inner[1]}'
    return str(s)

print('✅ Helpers definidos')


## Celda 5 — Construir repositorio (solo extrae, no imprime)


In [ ]:
repositorio = {}

CAMPOS_GRUPO = ['label', 'order', 'name', 'pretty', 'prim', 'solv', 'cyc', 'ab',
                'parity', 'nilpotency', 'num_conj_classes', 'gapidfull', 'abstract_label',
                'subfields', 'siblings', 'quotients', 'arith_equiv', 'transitivity',
                'gens', 'auts', 'malle_ainv', 'malle_b', 'malle_c', 'malle_d']

CAMPOS_POLYS = ['label', 'coeffs', 'disc_abs', 'disc_sign', 'r2',
                'ramps', 'num_ram', 'rd', 'grd', 'class_number',
                'regulator', 'galois_label', 'is_galois', 'cm',
                'conductor', 'narrow_class_number', 'used_grh']

print(f'=== Construyendo repositorio S{DEGREE} ===')
print(f'    Polinomios: {"TODOS" if MAX_POLYS is None else MAX_POLYS} por grupo')
print()

for t in range(1, MAX_T + 1):
    label = f'{DEGREE}T{t}'
    print(f'--- {label} ---')

    # Metadatos del grupo
    meta = db.gps_transitive.lucky({'n': DEGREE, 't': t}, CAMPOS_GRUPO)
    if meta:
        orden  = meta.get('order', '?')
        nombre = meta.get('name', '?')
        print(f'  ✅ grupo: orden={orden}  nombre={nombre}')
    else:
        meta = {}
        print(f'  ❌ sin metadatos')

    # Todos los polinomios (sin límite si MAX_POLYS es None)
    kwargs = {'query': {'degree': DEGREE, 'galt': t},
              'projection': CAMPOS_POLYS,
              'sort': ['disc_abs']}
    if MAX_POLYS is not None:
        kwargs['limit'] = MAX_POLYS

    polys = list(db.nf_fields.search(**kwargs))
    print(f'  📦 {len(polys)} polinomios')

    repositorio[label] = {
        'label'         : label,
        'degree'        : DEGREE,
        't_index'       : t,
        'group_metadata': meta,
        'polynomials'   : polys,
    }
    print()

total = sum(len(v['polynomials']) for v in repositorio.values())
print(f'✅ Repositorio completo: {len(repositorio)} grupos, {total} polinomios totales')


## Celda 6 — `get_group(dTx)` (función disponible, no se ejecuta aquí)


In [ ]:
def get_group(label):
    """
    Repositorio principal.
    Uso: get_group('7T3')

    Imprime en orden:
      1) Encabezado con nombre y label
      2) TABLA de polinomios irreducibles mónicos (una fila por polinomio)
      3) Toda la información contextual del grupo devuelta por LMFDB
    """
    label = label.strip().upper().replace(' ', '')

    # Descarga dinámica si no está en caché
    if label not in repositorio:
        print(f'⚠  {label} no en caché — descargando…')
        try:
            d_str, t_str = label.split('T')
            d, t = int(d_str), int(t_str)
        except Exception:
            print('❌ Formato inválido. Usa "7T3", "5T2", etc.')
            return
        meta  = db.gps_transitive.lucky({'n': d, 't': t}, CAMPOS_GRUPO) or {}
        kwargs = {'query': {'degree': d, 'galt': t},
                  'projection': CAMPOS_POLYS, 'sort': ['disc_abs']}
        if MAX_POLYS is not None:
            kwargs['limit'] = MAX_POLYS
        polys = list(db.nf_fields.search(**kwargs))
        repositorio[label] = {'label': label, 'degree': d, 't_index': t,
                               'group_metadata': meta, 'polynomials': polys}

    entry = repositorio[label]
    meta  = entry.get('group_metadata') or {}
    polys = entry.get('polynomials', [])
    deg   = entry.get('degree', DEGREE)

    # ═══════════════════════════════════════════════════════════════════════════
    # 1) ENCABEZADO
    # ═══════════════════════════════════════════════════════════════════════════
    nombre = meta.get('name', '?')
    order  = meta.get('order', '?')
    print()
    print('╔' + '═'*68 + '╗')
    print(f'║  GRUPO DE GALOIS: {label}   —   {nombre}'.ljust(69) + '║')
    print(f'║  Orden: {order}'.ljust(69) + '║')
    print('╚' + '═'*68 + '╝')

    # ═══════════════════════════════════════════════════════════════════════════
    # 2) TABLA DE POLINOMIOS MÓNICOS IRREDUCIBLES
    # ═══════════════════════════════════════════════════════════════════════════
    print()
    print(f'  ▶ POLINOMIOS MÓNICOS IRREDUCIBLES  f(x)  con  Gal(f) = {label}')
    n_total = len(polys)
    limite  = f'(mostrando {"TODOS" if MAX_POLYS is None else MAX_POLYS}: {n_total} registros, ordenados por |disc| ↑)'
    print(f'    {limite}')
    print()

    if not polys:
        print('    (Sin polinomios disponibles en LMFDB para este grupo)')
    else:
        W = 52
        print(f'  {"#":>4}  {"f(x)":<{W}}  {"|disc|":>18}  {"sgn":>4}  firma    LMFDB')
        print(f'  {"─"*4}  {"─"*W}  {"─"*18}  {"─"*4}  {"─"*7}  {"─"*30}')
        for i, p in enumerate(polys, 1):
            poly_str = coeffs_to_poly_str(p.get('coeffs', []))
            disc     = p.get('disc_abs', '?')
            sgn      = '+' if p.get('disc_sign', 1) == 1 else '−'
            r2       = p.get('r2')
            nf_lbl   = p.get('label', '')
            r1       = deg - 2*r2 if isinstance(r2, int) else '?'
            firma    = f'[{r1},{r2}]' if isinstance(r2, int) else '[?,?]'
            url      = f'https://www.lmfdb.org/NumberField/{nf_lbl}' if nf_lbl else ''
            if len(poly_str) > W:
                poly_str = poly_str[:W-1] + '…'
            print(f'  {i:>4}  {poly_str:<{W}}  {str(disc):>18}  {sgn:>4}  {firma:<7}  {url}')

    # ═══════════════════════════════════════════════════════════════════════════
    # 3) INFORMACIÓN CONTEXTUAL COMPLETA DEL GRUPO (todo lo que devuelve LMFDB)
    # ═══════════════════════════════════════════════════════════════════════════
    print()
    print('  ─'*35)
    print(f'  ▶ INFORMACIÓN CONTEXTUAL DEL GRUPO  {label}  (fuente: LMFDB gps_transitive)')
    print('  ─'*35)

    prim   = '✔ Sí' if meta.get('prim')    else '✘ No'
    solv   = '✔ Sí' if meta.get('solv')    else '✘ No'
    cyc    = '✔ Sí' if meta.get('cyc')     else '✘ No'
    ab     = '✔ Sí' if meta.get('ab')      else '✘ No'
    parity = 'Par (⊂ A_d)' if meta.get('parity', 0) == 1 else 'Impar'
    nilp   = meta.get('nilpotency', '?')
    nilp_s = str(nilp) if nilp != -1 else 'No nilpotente'
    nconj  = meta.get('num_conj_classes', '?')
    gapid  = meta.get('gapidfull', '?')
    abstrl = meta.get('abstract_label', '?')
    aeq    = meta.get('arith_equiv', '?')

    print(f'  Orden                  : {order}')
    print(f'  Paridad                : {parity}')
    print(f'  Primitivo              : {prim}')
    print(f'  Resoluble (solvable)   : {solv}')
    print(f'  Cíclico                : {cyc}')
    print(f'  Abeliano               : {ab}')
    print(f'  Nilpotencia            : {nilp_s}')
    print(f'  # Clases de conjugación: {nconj}')
    print(f'  GAP id                 : {gapid}')
    print(f'  Grupo abstracto        : {abstrl}')
    print(f'  Equiv. aritmética      : {aeq}')

    # Subcampos
    sf = meta.get('subfields', [])
    if sf:
        sf_str = ', '.join(_fmt_group_ref(s) for s in sf)
        print(f'  Subcampos              : {sf_str}')
    else:
        print(f'  Subcampos              : (ninguno)')

    # Grupos hermanos (misma acción sobre distintos conjuntos)
    sib = meta.get('siblings', [])
    if sib:
        sib_str = ', '.join(_fmt_group_ref(s) for s in sib)
        print(f'  Grupos hermanos        : {sib_str}')
    else:
        print(f'  Grupos hermanos        : (ninguno)')

    # Cocientes
    qt = meta.get('quotients', [])
    if qt:
        def _fmt_qt(q):
            if isinstance(q, (list, tuple)) and len(q) >= 2:
                return f'ord={q[0]}'
            return str(q)
        qt_str = ', '.join(_fmt_qt(q) for q in qt[:8])
        sufijo = f'  (+{len(qt)-8} más)' if len(qt) > 8 else ''
        print(f'  Cocientes (quot.)      : {qt_str}{sufijo}')
    else:
        print(f'  Cocientes              : (ninguno)')

    # Enlace LMFDB
    print(f'  URL LMFDB              : https://www.lmfdb.org/GaloisGroup/{label}')

    print()
    print('═'*70)
    print()


print('✅ get_group() lista.')
print()
print('  Uso: get_group("7T1")  …  get_group("7T7")')
print('       get_group("5T3")  — descarga dinámica para otros grados')


## Celda 7 — Exportar a JSON


In [ ]:
import json, pathlib

OUTPUT = f'galois_S{DEGREE}_COMPLETO.json'
with open(OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(repositorio, f, ensure_ascii=False, indent=2, default=str)

size_kb = pathlib.Path(OUTPUT).stat().st_size / 1024
print(f'✅ {OUTPUT}  ({size_kb:.1f} KB)')
resumen = {k: len(v["polynomials"]) for k, v in repositorio.items()}
print(f'   Polinomios por grupo: {resumen}')
print(f'   Total: {sum(resumen.values())} polinomios')


## Celda 8 — Descargar JSON


In [ ]:
from google.colab import files
files.download(OUTPUT)
print('📥 Descarga iniciada.')


## 📌 Subir el JSON a una base de datos para consultas

El archivo `galois_S7_COMPLETO.json` queda en el sistema de archivos de Colab y se descarga con la celda anterior. Para hacer consultas **sin volver a ejecutar este notebook**, súbelo a una de estas opciones:

| Opción | Cuándo usarla | Cómo |
|---|---|---|
| **MongoDB Atlas** (free 512 MB) | Consultas tipo `{group:'7T3', disc_abs:{$lt:1e6}}` | `pymongo` + `insert_many` por grupo |
| **SQLite local** | Consultas SQL simples, cero configuración | `sqlite3` + `json_each` o aplanar a tabla |
| **Google Sheets / BigQuery** | Compartir con otros, análisis ad-hoc | Importar JSON directo |
| **Supabase (Postgres gratis)** | Si luego quieres una API REST automática | sube vía su dashboard o `psycopg2` |

**Recomendación para tu caso** (solo necesitas consultar, no más cómputo pesado): **MongoDB Atlas**. Cada documento puede ser exactamente una entrada de tu JSON (`{label, degree, t_index, group_metadata, polynomials}`), así que la subida es literalmente `db.galois_groups.insert_many(list(repositorio.values()))` sin transformar nada.

Ejemplo de subida (ejecutar fuera de Colab, o en una celda aparte si tienes tu URI de conexión):
```python
from pymongo import MongoClient
import json

client = MongoClient('TU_URI_DE_MONGODB_ATLAS')
col = client['galois_db']['s7_groups']

with open('galois_S7_COMPLETO.json', encoding='utf-8') as f:
    data = json.load(f)

col.insert_many(list(data.values()))
print(f'✅ {col.count_documents({})} grupos subidos a MongoDB')
```

Luego, desde cualquier script (sin Colab, sin LMFDB), consultas así:
```python
# Todos los polinomios de 7T3
doc = col.find_one({'label': '7T3'})
polinomios = doc['polynomials']
```
